# ESM2 Embeddings and ESMFold

**Tier 5 — Modern AI for Science | Module 06 · Notebook 1**

*Prerequisites: Module 04 (AlphaFold), basic PyTorch/transformers familiarity*

---

**By the end of this notebook you will be able to:**
1. Generate protein sequence embeddings
2. Build a simple downstream function probe
3. Understand ESMFold usage and confidence outputs
4. Compare embedding and structure-centric workflows

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook demonstrates how foundation models are used in modern computational biology for representation learning, prioritization, and hypothesis generation.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Model score != biological truth. Treat predictions as ranked hypotheses requiring validation.
- Context length and tokenization can change model behavior more than minor hyperparameter tweaks.
- Domain shift is common: performance on curated benchmarks may not transfer to your assay/data source.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

# CSV preview (DNA)
with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

# Variants preview
with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

# FASTA preview
print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

# Metadata preview
meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Module 05](../05_Genomic_Foundation_Models/) | [Module Overview](README.md) | [Next: Zero-Shot Mutation →](./02_zero_shot_mutation.ipynb)

---

In [ ]:
import numpy as np

np.random.seed(9)

## 1. Embedding Idea

Protein language models map sequences to dense vectors where nearby vectors often share structural or functional properties.

In this notebook we keep examples lightweight and demonstrate workflow mechanics.

In [ ]:
AA = list('ACDEFGHIKLMNPQRSTVWY')
AA_INDEX = {a: i for i, a in enumerate(AA)}

def toy_embed(seq: str) -> np.ndarray:
    # 20-aa composition + simple length feature
    vec = np.zeros(21, dtype=float)
    for a in seq:
        if a in AA_INDEX:
            vec[AA_INDEX[a]] += 1
    if len(seq) > 0:
        vec[:20] /= len(seq)
    vec[20] = min(len(seq) / 500.0, 1.0)
    return vec

seq = 'MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ'
emb = toy_embed(seq)
print('Embedding shape:', emb.shape)
print('Top composition entries:', np.argsort(emb[:20])[-5:][::-1])

## 2. Downstream Probe: Binary Classification

Synthetic task: classify sequences by presence of a motif correlated with class 1.

In [ ]:
def random_protein(n: int) -> str:
    return ''.join(np.random.choice(AA, size=n))

def inject_motif(seq: str, motif: str, pos: int) -> str:
    return seq[:pos] + motif + seq[pos+len(motif):]

N = 200
motif = 'HGH'
seqs, y = [], []
for _ in range(N):
    s = random_protein(80)
    if np.random.rand() < 0.5:
        s = inject_motif(s, motif, 25)
        y.append(1)
    else:
        y.append(0)
    seqs.append(s)

X = np.vstack([toy_embed(s) for s in seqs])
y = np.array(y)

train, test = np.arange(150), np.arange(150, N)
c0 = X[train][y[train] == 0].mean(axis=0)
c1 = X[train][y[train] == 1].mean(axis=0)
pred = (((X[test]-c1)**2).sum(axis=1) < ((X[test]-c0)**2).sum(axis=1)).astype(int)
print('Probe accuracy:', float((pred == y[test]).mean()))

## 3. ESMFold Usage Pattern

ESMFold provides fast structure predictions without explicit MSA generation.

Typical outputs include atom coordinates and confidence values (pLDDT-like signals).

In [ ]:
# Optional real API usage (network required)
# import requests
# sequence = 'MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ'
# r = requests.post('https://api.esmatlas.com/foldSequence/v1/pdb/', data=sequence)
# with open('esmfold_prediction.pdb', 'w') as f:
#     f.write(r.text)

def confidence_bucket(mean_plddt: float) -> str:
    if mean_plddt >= 90:
        return 'very_high'
    if mean_plddt >= 70:
        return 'usable'
    if mean_plddt >= 50:
        return 'low'
    return 'very_low'

for v in [96, 82, 63, 41]:
    print(v, confidence_bucket(v))

## 4. Practical Guidance

- Use embeddings for large-scale screening and annotation tasks.
- Use structure prediction when mechanism likely depends on 3D context.
- Keep confidence-aware filters to avoid overinterpretation.

## Summary

- Embeddings provide compact sequence representations for downstream ML.
- ESMFold offers fast structure hypotheses.
- Combining embedding and structure workflows is often more robust than either alone.

## Source-backed Context

- ESM repository documentation groups ESM-2, ESMFold, ESM-1v, and ESM-IF1 as a coherent protein LM ecosystem.
- ESMFold is used in practice as a fast MSA-free structural hypothesis generator.


## Validated Sources

Checked online during content expansion.

- [ESM official repository](https://github.com/facebookresearch/esm)
- [ESM Atlas](https://esmatlas.com)
